# Five-fold development validation of A/B and classical controls

## Objective
Inspect the completed, fixed 50-epoch development experiment (five folds × three seeds). This notebook reads aggregate summaries only, never fits models, and never reads cohort data or checkpoints. The 15 results per model are dependent observations. See the [self-contained report](../reports/phase2/cv_5fold_50e_v1/report.md) for methodology, numerical conclusions, and limitations.

B improves ROC-AUC over A in 14 of 15 paired results, with one tie; its mean ROC-AUC is 0.9215 versus 0.8297. The frozen probe exceeds B in 12 pairs. B has numerical rank below four in 9 of 15 final embeddings, occurring in all five folds. These counts are descriptive, not independent statistical replicates.

## Load existing artifacts
Run from the repository root or `notebooks/`. Set `QARL_CV_ANALYSIS_DIR` to another verified summary directory if needed. Dependencies are the existing analysis and notebook extras in `pyproject.toml`. Detailed private artifacts remain under `runs/phase2/`; these figures need only the small aggregate CSVs.

In [ ]:
import os
from pathlib import Path
import sys
import matplotlib.pyplot as plt
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
if not (root / "src/phase2").is_dir():
    raise RuntimeError("Run from the repository root or notebooks directory.")
sys.path.insert(0, str(root))
from src.phase2.cv_views import (load_tables, plot_metrics, plot_pairs, plot_spectrum,
    plot_coordinates, plot_learning, plot_generalization, plot_reliability)
analysis = Path(os.environ.get("QARL_CV_ANALYSIS_DIR", str(root / "reports/phase2/cv_5fold_50e_v1")))
tables = load_tables(analysis)
plt.rcParams.update({"font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})


## Diagnostics and visualizations
### Predictive performance
Higher ROC-AUC, class-specific AP, and balanced accuracy are better; lower log loss is better. ER+=1 and ER−=0. Black marks denote seed means within folds. The spread shown is descriptive, not a confidence interval.

In [ ]:
display(tables["summary"].round(4))
plot_metrics(tables["metrics"])
plt.show()

### Paired B−A differences
Rows preserve the five validation partitions and columns preserve the original seeds. Color ranges differ across metrics. Positive differences favor B for discrimination/classification; negative differences favor B for log loss.

In [ ]:
plot_pairs(tables["paired"], "B_minus_A")
plt.show()
display(tables["paired_fold_means"][["fold", "B_minus_A_roc_auc", "B_minus_B_frozen_probe_roc_auc"]].round(4))

### B versus the frozen-encoder linear probe
This comparison changes the head after B's encoder was trained jointly with the VQC. It tests linear readability of that representation, without identifying the VQC's causal contribution to learning it. Ranking, threshold decisions, and probability loss may disagree.

In [ ]:
plot_pairs(tables["paired"], "B_minus_B_frozen_probe")
plt.show()

### Generalization at the same final checkpoint
These gaps reconstruct training predictions from saved final latent arrays and heads. They differ from the online training losses in the historical learning curves. Positive loss gaps and negative AUC gaps describe a training/validation separation; they do not identify its cause.

In [ ]:
plot_generalization(tables["generalization"])
plt.show()

### Learning trajectories
Solid curves are validation BCE and dashed curves are online training BCE. Each line averages the three seeds within a fold. Vertical scales differ by model. All 50 epochs are retained; no minimum is used for model selection.

In [ ]:
plot_learning(tables["curves"], "loss")
plt.show()

### Latent spectrum and coordinate variability
Spectra are centered and normalized by their largest singular value. Numerical rank uses a relative threshold of 1e−6; the display floor is 1e−12. Entropy effective rank uses variance weights s²/Σs² and is a distinct continuous descriptor. Coordinate SD is in pre-angular latent units, ddof=0; a small SD may correspond to an almost constant input. Full rank alone does not establish a useful representation.

In [ ]:
plot_spectrum(tables["latent"])
plt.show()
plot_coordinates(tables["latent"])
plt.show()

### Angular saturation
Saturation is the fraction of patient × coordinate values with |tanh(z/2)|>0.95. The CSV also retains the >0.99 fraction and derivative <0.01, excluding π. B_classical does not apply this angular transformation: its saturation values are hypothetical. Vertical scales differ by model.

In [ ]:
plot_learning(tables["curves"], "saturation")
plt.show()

### Gradients and optimization stability
Head and encoder norms are mean minibatch L2 norms, averaged over seeds within each fold. Nonzero gradients show connection to the loss, without proving predictive usefulness. Final-layer RZ rotations are structurally redundant and remain in the frozen architecture. The raw curve summaries retain all parameter-wise RMS gradients and seed trajectories.

In [ ]:
plot_learning(tables["curves"], "gradient")
plt.show()

### Descriptive probability reliability
Ten fixed equal-width probability bins summarize out-of-fold predictions. Each patient contributes one validation prediction per seed; pooling therefore repeats patients three times. These points have no independent-sample confidence intervals and no calibrator is fitted. Log loss and Brier score reflect both discrimination and calibration, rather than isolating calibration.

In [ ]:
plot_reliability(tables["calibration"])
plt.show()

## Interpretation
Mean validation log loss is 0.2867 for B, 0.2819 for the probe, and 0.7585 for B_classical. B retains rank four in six runs, rank three in eight, and rank two in one. B_classical retains numerical rank four but has lower variance entropy rank than B, so numerical rank and effective dimensionality must be distinguished. Read the report's numerical comparisons alongside these views. Examine each fold and seed before averaging. Compare B with both the jointly trained classical-head control and the post-training probe; neither comparison by itself isolates a quantum contribution. Coordinate contraction, saturation, and generalization gaps are descriptive observations, not identified causal mechanisms.

## Limitations
This is development CV after an exploratory pilot, with overlapping training sets and three repeated seeds. It is not an independent confirmation cohort or a locked-test evaluation. Timing is CPU simulation with heterogeneous historical/new execution conditions and excludes some pipeline stages. No quantum superiority, methodological novelty, or causal role for a regularizer is established. C0/C1 remain unimplemented.